# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [1]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [2]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [3]:
# TODO
og_len = len(df)
df = df.drop_duplicates()
dropped = og_len - len(df)
log('drop_duplicates', 'Dropped duplicate rows', dropped)

[drop_duplicates] Dropped duplicate rows (15 row(s))


### TODO 2 — clean `price` -> float

In [4]:
# TODO
df["price"] = df["price"].str.replace("$", "").astype(float)
log("Clean prices", "Stripped '$' and cast price to float", len(df))

[Clean prices] Stripped '$' and cast price to float (300 row(s))


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [5]:
# TODO
before_qty = len(df)
df = df[df["qty"] > 0].copy()
df["qty"] = df["qty"].astype(int)

log("dropped missing or neg qty", "Dropped missing and negative quantities", before_qty - len(df))

[dropped missing or neg qty] Dropped missing and negative quantities (25 row(s))


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [6]:
# TODO: inspect the variants, build a mapping dict, apply it, log the collapse
print("before map:\n", df["item"].value_counts())
ITEM_MAP = {
    "Cheeseburger": "Cheeseburger",
    "cheese burger": "Cheeseburger",
    "Foam Finger": "Foam Finger",
    "foam finger": "Foam Finger",
    "Rain Poncho": "Rain Poncho",
    "rain poncho": "Rain Poncho"
}
df["item"] = df["item"].map(ITEM_MAP)

log( 'canonicalize_item', 'Canonicalized item names to 3 products', len(df),)

before map:
 item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
[canonicalize_item] Canonicalized item names to 3 products (275 row(s))


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [7]:
# TODO
CAT_MAP = {
    "Food": "Food",
    "food": "Food",
    "Merch": "Merchandise",
    "Apparel": "Merchandise",
    "RainGear": "Rain Gear",
    "rain-gear": "Rain Gear"
}

ITEM_TO_CAT = {
    "Cheeseburger": "Food",
    "Foam Finger": "Merchandise",
    "Rain Poncho": "Rain Gear"
}

df["category"] = df["item"].map(ITEM_TO_CAT)
log('normalize_category', 'Assigned canonical category based on item type', len(df),)

[normalize_category] Assigned canonical category based on item type (275 row(s))


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [8]:
assert df.duplicated().sum() == 0
assert df['qty'].min() >= 1
assert df['price'].dtype == float
# TODO: assert something about item
valid_items = {"Cheeseburger", "Foam Finger", "Rain Poncho"}
assert set(df["item"].unique()) == valid_items, "Unexpected names found"
# TODO: assert something about category
assert df["category"].isna().sum() == 0, "Missing categories detected"
assert set(df["category"].unique()).issubset({"Food", "Merchandise", "Rain Gear"}), "Invalid category found"
print('clean:', df.shape)

clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [9]:
# TODO
df["revenue"] = df["qty"] * df["price"]

rev_by_cat = df.groupby("category")["revenue"].sum().sort_values(ascending=False).round(2)
print("Revenue by Category:")
print(rev_by_cat)
print("-" * 25)
print(f"Total Revenue: ${df['revenue'].sum():.2f}")

Revenue by Category:
category
Rain Gear      1729.5
Merchandise    1593.0
Food           1417.5
Name: revenue, dtype: float64
-------------------------
Total Revenue: $4740.00


**What I would tell the vendor:** _.I would tell the vendors to stock up on more of Raingear as that category produced the most amount of revenue and slowly, if needed, to phase out underperforming stocks._

### TODO 8 — read back your log

In [11]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,drop_duplicates,Dropped duplicate rows,15
1,Clean prices,Stripped '$' and cast price to float,300
2,dropped missing or neg qty,Dropped missing and negative quantities,25
3,canonicalize_item,Canonicalized item names to 3 products,275
4,normalize_category,Assigned canonical category based on item type,275


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.



**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.


_**Answer A:** Step 3 or todo 3;
before drop: 4594.5 after drop: 4740.0_

**Answer B:** A decision I made was dropping the negative value entirely when it couldnt been treated as customer refunds or returns. Not sure about the scope of this assignment but that is plausible. Since we dropped them, the gross revnue was, as we stated above 4740.00, though if we treated them as refunds we would've had 4596.50 as the new revenue. I decided to go with dropping the items as in a stadium log data with no tracking id, these could just be corrupted.